# Pipeline Logging
**Purpose:** Record pipeline run metadata — row counts, timestamps, compliance summary — for audit and monitoring.
**Writes to:** gold.pipeline_run_log (append mode — full history preserved)

In [ ]:
# Catalog widget — injected by Job base_parameters or set manually
dbutils.widgets.text("catalog", "metadata_governance", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

## Step 1 — Collect Run Metrics from All Layers

In [ ]:
from pyspark.sql.functions import current_timestamp, current_date
from pyspark.sql import Row

bronze_count = spark.table(f"{catalog}.bronze.bronze_metadata").count()
silver_count = spark.table(f"{catalog}.silver.silver_metadata").count()

try:
    quarantine_count = spark.table(f"{catalog}.silver.bronze_metadata_quarantine").count()
except:
    quarantine_count = 0

gold_kpis = spark.table(f"{catalog}.gold.gold_compliance_kpis").collect()[0]

print(f"Bronze rows:     {bronze_count}")
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")
print(f"Compliance rate: {gold_kpis.compliance_rate_pct}%")

## Step 2 — Write Pipeline Run Log

In [ ]:
log_row = Row(
    catalog=catalog,
    bronze_row_count=bronze_count,
    silver_row_count=silver_count,
    quarantine_row_count=quarantine_count,
    total_tables=int(gold_kpis.total_tables),
    compliant_tables=int(gold_kpis.compliant_tables),
    non_compliant_tables=int(gold_kpis.non_compliant_tables),
    compliance_rate_pct=float(gold_kpis.compliance_rate_pct),
    pipeline_status="SUCCESS"
)

log_df = spark.createDataFrame([log_row]) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_date", current_date())

# Append — keeps full history of every run
log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog}.gold.pipeline_run_log")

print("Pipeline run logged successfully.")
display(log_df)

## Step 3 — Display Full Run History

In [ ]:
display(
    spark.table(f"{catalog}.gold.pipeline_run_log")
    .orderBy("run_timestamp", ascending=False)
)